In [0]:
%pip install geopandas shapely pyproj fiona
dbutils.library.restartPython()

In [0]:
import geopandas as gpd
import pandas as pd
from shapely.wkt import loads as wkt_loads
from shapely.geometry import Point
from pyspark.sql.functions import col

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA silver")

In [0]:
# We stored geometry as WKT strings in bronze.
# shapely.wkt.loads() converts each string back into a proper geometry object.
# We only keep the columns we need for the join and the output.
df_catchments = spark.table("scottish_housing_ws.bronze.school_catchments").toPandas()

df_catchments["geometry"] = df_catchments["geometry_wkt"].apply(
    lambda wkt: wkt_loads(wkt) if wkt else None
)

gdf_catchments = gpd.GeoDataFrame(
    df_catchments[["la_s_code", "local_auth", "seed_code", "school_nam", "geometry"]],
    geometry="geometry",
    crs="EPSG:27700"
)

print(f"Catchment polygons loaded: {len(gdf_catchments)}")
gdf_catchments.head(3)

In [0]:
# easting and northing are in EPSG:27700, matching the catchment polygons.
# We filter out any postcodes with null coordinates -- a small number may
# have no grid reference in the source data.
df_postcodes = (
    spark.table("scottish_housing_ws.silver.postcode_lookup")
    .select("postcode", "council_area_code", "easting", "northing")
    .filter(col("easting").isNotNull() & col("northing").isNotNull())
    .toPandas()
)

# Convert easting/northing to Point geometry objects
df_postcodes["geometry"] = df_postcodes.apply(
    lambda row: Point(row["easting"], row["northing"]), axis=1
)

gdf_postcodes = gpd.GeoDataFrame(
    df_postcodes,
    geometry="geometry",
    crs="EPSG:27700"
)

print(f"Postcodes loaded: {len(gdf_postcodes)}")
gdf_postcodes.head(3)

In [0]:
# gpd.sjoin finds which catchment polygon each postcode point falls inside.
# how="left" keeps all postcodes, including those that don't fall inside any
# catchment (e.g. remote postcodes with no defined catchment boundary).
# predicate="within" tests whether the point is within the polygon.
# This is the core operation -- 161k points against 328 polygons.
# It will take a minute or two to run.

gdf_joined = gpd.sjoin(
    gdf_postcodes,
    gdf_catchments,
    how="left",
    predicate="within"
)

print(f"Rows after join: {len(gdf_joined)}")
gdf_joined.head(5)

In [0]:
matched = gdf_joined["school_nam"].notna().sum()
total = len(gdf_joined)
print(f"Postcodes matched to a catchment: {matched:,} of {total:,} ({matched/total*100:.1f}%)")

In [0]:
df_silver = (
    spark.createDataFrame(
        gdf_joined[[
            "postcode",
            "council_area_code",
            "seed_code",
            "school_nam",
            "local_auth"
        ]].rename(columns={
            "school_nam": "secondary_school_name",
            "local_auth": "school_local_authority",
            "seed_code": "seed_code"
        })
    )
    .withColumn("seed_code", col("seed_code").cast("long"))
)

df_silver.printSchema()

In [0]:
df_silver.filter(col("council_area_code") == "S12000036").show(20, truncate=False)

In [0]:
from pyspark.sql.functions import count
df_silver.agg(count("*").alias("total_postcodes")).show()

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.silver.school_catchments")
)

In [0]:
# Postcode to secondary school catchment lookup for all Scottish postcodes.
# Point-in-polygon spatial join between postcode centroids (EPSG:27700)
# and school catchment polygons from Spatial Hub Scotland.
# seed_code is the unique school identifier -- joins to attainment data in gold.
# Postcodes with no matched catchment will have null seed_code and school name
# -- expected for remote areas with incomplete catchment boundary data.
# 328 secondary non-denominational catchment polygons across 32 councils.